## 1. Install and import the libraries

One install line covers all the libraries used in this notebook.
Restart the kernel after installing if asked.

In [2]:
%pip install -q numpy pandas torch transformers sentence-transformers gensim openai accelerate ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import numpy as np
import pandas as pd
import torch
import transformers

## 2. Static embeddings: GloVe

Every word has one fixed vector, whatever sentence it is in.
Load the model and look at the vector of 'server'.

In [4]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-50")

print("vocabulary size:", len(glove.key_to_index))
print("vector size    :", glove.vector_size)
print("vector of 'server' (first 6 numbers):", glove["server"][:6].round(2))

vocabulary size: 400000
vector size    : 50
vector of 'server' (first 6 numbers): [ 0.77 -0.38  1.12  1.77  0.08 -0.89]


## 3. One vector for 'bank'

The nearest words to 'bank' are all about money, not rivers.
A static model cannot separate the two meanings.

In [5]:
print("15 nearest to 'bank':", [w for w, _ in glove.most_similar("bank", topn=15)])
print()
for word in ["loan", "money", "river", "water"]:
    print(f"bank vs {word:6s} cosine = {glove.similarity('bank', word):.2f}")

15 nearest to 'bank': ['banks', 'securities', 'banking', 'investment', 'exchange', 'financial', 'credit', 'lender', 'capital', 'brokerage', 'currency', 'investments', 'funds', 'fund', 'assets']

bank vs loan   cosine = 0.68
bank vs money  cosine = 0.64
bank vs river  cosine = 0.41
bank vs water  cosine = 0.34


## 4. Static embeddings ignore word order

'dog bites man' and 'man bites dog' get the same score of 1.0.
The meaning is different, but the average of the words is the same.

In [6]:
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
def glove_sentence_vector(sentence):
    return np.mean([glove[word] for word in sentence.lower().split()], axis=0)

a = glove_sentence_vector("dog bites man")
b = glove_sentence_vector("man bites dog")
print("'dog bites man' vs 'man bites dog': cosine =", round(cosine(a, b), 2))

'dog bites man' vs 'man bites dog': cosine = 1.0


## 5. Contextual embeddings: DistilBERT

A transformer gives the same word a different vector depending on the sentence.
'bank' in a river sentence is not the same as 'bank' in a loan sentence.

In [7]:
from transformers import AutoTokenizer, AutoModel

bert_name = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert_model = AutoModel.from_pretrained(bert_name)
bert_model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [8]:
def bank_contextual(text):
    enc = bert_tokenizer(text, return_tensors="pt")
    with torch.no_grad():                                        # prediction only, no gradients needed
        hidden = bert_model(**enc).last_hidden_state[0]          # one vector per token, after the layers
    position = bert_tokenizer.convert_ids_to_tokens(enc["input_ids"][0]).index("bank")
    return hidden[position].numpy()

river1 = bank_contextual("the fisherman sat on the bank of the river")
river2 = bank_contextual("we walked along the muddy river bank")
money = bank_contextual("the bank approved my home loan")

print("GloVe (static)                  : 'bank' vs 'bank' = 1.00 (always the same vector)")
print("transformer: river  vs river2  :", round(cosine(river1, river2), 2))
print("transformer: river bank 1 vs money bank:", round(cosine(river1, money), 2))
print("transformer: river bank 2 vs money bank:", round(cosine(river2, money), 2))

GloVe (static)                  : 'bank' vs 'bank' = 1.00 (always the same vector)
transformer: river  vs river2  : 0.77
transformer: river bank 1 vs money bank: 0.58
transformer: river bank 2 vs money bank: 0.55


## 6. Sentence embeddings: SentenceTransformer

It turns a whole sentence into one vector, so we can compare sentences by meaning.
Sentences about the same problem score higher than unrelated ones.

In [9]:
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentences = {
    "A1": "pods keep restarting after the release",
    "A2": "container crash loop following the deployment",
    "B1": "disk is almost full on the database server",
    "B2": "no storage space left on the DB host",
    "C":  "my favourite pizza has extra cheese",
}
vectors = st_model.encode(list(sentences.values()), normalize_embeddings=True)      # (5, 384), every row has length 1
similarity = vectors @ vectors.T                                                     # dot product of unit vectors = cosine

print(pd.DataFrame(similarity, index=sentences.keys(), columns=sentences.keys()).round(2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

      A1    A2    B1    B2     C
A1  1.00  0.43  0.13  0.15 -0.10
A2  0.43  1.00  0.12  0.23 -0.06
B1  0.13  0.12  1.00  0.69  0.05
B2  0.15  0.23  0.69  1.00  0.02
C  -0.10 -0.06  0.05  0.02  1.00


## 7. DevOps use case: match an alert to a runbook

Embed the runbook titles and the alert, then rank the runbooks by similarity.
The best match is the runbook that fits the alert.

In [10]:
runbooks = [
    "Fix CrashLoopBackOff: pod keeps restarting",
    "Disk full: clean up logs and expand the volume",
    "High CPU on a node: find the noisy process",
    "Certificate expired: renew the TLS certificate",
    "Database connection pool exhausted",
]
runbook_vectors = st_model.encode(runbooks, normalize_embeddings=True)

alert = "containers restart over and over after the new image was deployed"
scores = runbook_vectors @ st_model.encode(alert, normalize_embeddings=True)
for i in scores.argsort()[::-1]:
    print(f"{scores[i]:.2f}  {runbooks[i]}")

0.40  Fix CrashLoopBackOff: pod keeps restarting
0.22  Database connection pool exhausted
0.20  Certificate expired: renew the TLS certificate
0.10  Disk full: clean up logs and expand the volume
0.08  High CPU on a node: find the noisy process


## 8. Text generation with GPT-2: load the model

GPT-2 is a small decoder model that predicts the next token.
Load the tokenizer and the model, then switch to eval mode.

In [11]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [12]:
tok = GPT2Tokenizer.from_pretrained("gpt2")

In [13]:
model = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="eager") 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 9. Tokenize the input text

Turn the sentence into token IDs that the model accepts.
Return them as PyTorch tensors.

In [15]:
text = "Rose loves sour fresh fruits like strawberries, apples"

In [16]:
inputs = tok(text, return_tensors="pt")

In [17]:
inputs

{'input_ids': tensor([[31087, 10408, 11348,  4713, 15921,   588, 36973,    11, 22514]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

## 10. Run the model to get logits

The model gives one raw score (logit) for every token in its vocabulary.
The shape is (sentences, tokens, 50257).

In [18]:
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

In [19]:
logits

tensor([[[ -31.2852,  -30.6273,  -34.2245,  ...,  -39.2340,  -38.1299,
           -31.6271],
         [ -74.0125,  -75.1880,  -81.0360,  ...,  -81.6594,  -82.0551,
           -76.6324],
         [ -81.2202,  -82.9256,  -88.6227,  ...,  -90.3833,  -91.0548,
           -84.3656],
         ...,
         [ -61.7508,  -65.3426,  -70.5698,  ...,  -77.2291,  -74.8082,
           -66.6509],
         [ -98.4999, -101.2800, -103.2987,  ..., -107.5981, -103.5498,
          -100.3659],
         [ -21.2307,  -23.4124,  -27.5964,  ...,  -34.3731,  -31.4195,
           -25.1886]]])

In [20]:
print("logits shape:", logits.shape, "= (sentences, tokens, vocabulary size)")

logits shape: torch.Size([1, 9, 50257]) = (sentences, tokens, vocabulary size)


## 11. Logits to probabilities (softmax)

Take the scores for the last position, because that predicts the next token.
Softmax turns them into probabilities that add up to 1.

In [21]:
next_token_logits = logits[:, -1, :]

In [22]:
probs = torch.softmax(next_token_logits, dim=-1) 

In [23]:
probs

tensor([[9.0013e-05, 1.0158e-05, 1.5478e-07,  ..., 1.7646e-10, 3.3837e-09,
         1.7195e-06]])

## 12. Top 5 next-token predictions

topk picks the 5 most likely tokens.
decode turns each token ID back into text.

In [24]:
top_k = torch.topk(probs, k=5)  

In [25]:
top_k

torch.return_types.topk(
values=tensor([[0.6939, 0.2731, 0.0073, 0.0054, 0.0048]]),
indices=tensor([[  11,  290,  559,  393, 1222]]))

In [26]:
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tok.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

',' : 0.6939
' and' : 0.2731
'au' : 0.0073
' or' : 0.0054
' &' : 0.0048


## 13. Another example

The same steps in one cell, with a longer sentence.
The model gives ' happens' the highest chance, about 53%.

In [27]:
text = "The kids are watching this movie with their parents and they want to watch the movie and see what happens. They  want  to  see what"
inputs = tok(text, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits[:, -1, :], dim=-1)
top_k = torch.topk(probs, k=5)
for idx, score in zip(top_k.indices[0], top_k.values[0]):
    predicted_word = tok.decode([idx])
    print(f"{predicted_word!r} : {float(score):.4f}")

' happens' : 0.5299
' they' : 0.0764
' the' : 0.0437
' it' : 0.0334
' happened' : 0.0322


## 14. Generate text with Flan-T5 (greedy)

generate() repeats 'predict the next token' until the answer is complete.
The default picks the most likely token each time.

In [28]:
from transformers import AutoModelForSeq2SeqLM

t5_name = "google/flan-t5-base"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(**inputs, max_new_tokens=100, no_repeat_ngram_size=3)      # stops it repeating the same phrase
print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The story of a young man who has been a sailor for a long time. He is a very talented stag, who is very good at spotting the elusive elusive fish. He has a great sense of humor and is very lovable. He loves to go fishing and is able to get away from his stumbling blocks. He also loves to be around his family and friends. He likes to go to the


## 15. Add randomness: temperature

do_sample=True lets the model pick from the likely tokens instead of always the top one.
A higher temperature makes the story more varied.

In [29]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

t5_name = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(
    **inputs,
    max_new_tokens=100,
    no_repeat_ngram_size=3,
    do_sample=True,
    temperature=0.7
)

print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


This is my second love story. I found an old friend from college and I would go on a date to see her and she loved him. We were going out to the beach and at the beach, and I thought it was amazing that you could be in love with her. So, I decided to go out and have a romantic date with her and I am happy.


## 16. Limit the choices: top_k

top_k=50 means it chooses only from the 50 most likely tokens.
This keeps the randomness under control.

In [30]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

t5_name = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_name).to(
    "cuda" if torch.cuda.is_available() else "cpu"
)

prompt = "Can you write a short beautiful love story?"
inputs = t5_tokenizer(prompt, return_tensors="pt").to(t5_model.device)

output = t5_model.generate(
    **inputs,
    max_new_tokens=100,
    no_repeat_ngram_size=3,
    do_sample=True,
    temperature=0.7,
    top_k=50
)

print(t5_tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


A young woman is preparing for her wedding with a bridesmaid. The brides maids are preparing to leave the palace. The maid of honor is at the wedding. The wedding is being planned. The groom is expected to arrive. The couple is about to depart. The beautiful bride is in love with her.


LLMS Demo

## 17. Call OpenAI: set the key and the client

The key is read from an environment variable, or you type it in (hidden).
Never put a key in the notebook text or in GitHub.

In [32]:
import os
import getpass

openai_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Paste your OpenAI API key (hidden, not saved): ")

Paste your OpenAI API key (hidden, not saved):  ········


In [33]:
from openai import OpenAI

In [34]:
client = OpenAI(api_key=openai_key)

## 18. Send a prompt to GPT-5

The input is the prompt. max_output_tokens limits the answer length.
Low reasoning effort is faster and cheaper.

In [35]:
resp = client.responses.create(
    model="gpt-5",                       # use a newer model name here if your account lists one
    input="In two sentences, what is a Kubernetes pod?",
    reasoning={"effort": "low"},         # GPT-5 is a reasoning model: low effort = faster and cheaper
    max_output_tokens=100,              # the thinking tokens count too, so leave room
)

RateLimitError: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.

## 19. Call Anthropic: set the key and the client

Same idea: get the key, install the library and create the client.
The Anthropic library is called anthropic.

In [36]:
anthropic_key = os.getenv("ANTHROPIC_API_KEY") or getpass.getpass("Paste your Anthropic API key (hidden, not saved): ")

Paste your Anthropic API key (hidden, not saved):  ········


In [37]:
import anthropic

ModuleNotFoundError: No module named 'anthropic'

In [38]:
!python3 -m pip install anthropic

  Using cached anthropic-2.6.0-py3-none-any.whl.metadata (3.6 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached opentelemetry_api-1.45.2-py3-none-any.whl.metadata (1.4 kB)
Using cached anthropic-2.6.0-py3-none-any.whl (2.4 MB)
Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
Using cached opentelemetry_api-1.45.2-py3-none-any.whl (60 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [anthropic]/3 [anthropic]


In [39]:
import anthropic

In [40]:
claude_client = anthropic.Anthropic(api_key=anthropic_key)

## 20. Send a prompt to Claude

messages holds the prompt, and the role 'user' means it comes from the person.
max_tokens is required.

In [41]:
resp = claude_client.messages.create(
    model="claude-opus-5-5",
    max_tokens=100,                          # required; thinking tokens count too, so leave room
    output_config={"effort": "low"},          # less thinking = faster and cheaper
    messages=[{"role": "user", "content": "In two sentences, what is a Kubernetes pod?"}],
)

## 21. Read the response

The reply has an id, the model, the content, the stop reason and the token usage.
Pull out just the text block as the answer.

In [43]:
resp

Message(id='msg_011Cgkt315xLEek9FGKWCLqD', container=None, content=[TextBlock(citations=None, text='A Kubernetes pod is the smallest deployable unit in Kubernetes, consisting of one or more containers that share the same network namespace (IP address and ports) and storage volumes. Pods are scheduled together onto a single node and are typically managed by higher-level controllers like Deployments, which handle scaling,', type='text')], diagnostics=None, model='claude-opus-5-5', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=26, output_tokens=100, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

In [44]:
answer = next(block.text for block in resp.content if block.type == "text")  

In [45]:
answer

'A Kubernetes pod is the smallest deployable unit in Kubernetes, consisting of one or more containers that share the same network namespace (IP address and ports) and storage volumes. Pods are scheduled together onto a single node and are typically managed by higher-level controllers like Deployments, which handle scaling,'